# Project 1: VaR and ES -- Estimation, Backtesting, and Model Comparison

**Names of all group members:**
- Firstname Lastname (email@example.com)
- Firstname Lastname (email@example.com)

---

## 0) Setup

Creates necessary folders, sets parameters, and fixes the random seed. Keep this seed fixed; the whole run must be reproducible from it.

In [1]:
# If a package import fails, install it in your environment, e.g.:
# %pip install arch statsmodels

import os
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_DIR = Path.cwd()
DATA_FILE = PROJECT_DIR / "QRM_2026_Project_prices.csv"   # provided with the project; do not re-download
OUT_DIR = PROJECT_DIR / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Parameters
TICKERS = ["AAPL", "META", "JPM"]   # every question is run separately on each stock
WINDOW = 504                        # estimation window of every model, in trading days (Questions 2-5)
EVAL_START = 504                    # Questions 3-5: forecast days are all days after the first 504
REFIT = 21                          # Questions 4-5: re-estimate every this many trading days
ALPHAS = [0.975, 0.99]              # confidence levels used throughout Questions 2-5
DEEP_ALPHA = 0.999                  # Question 6: the extreme level EVT is asked for
M_SIM = 10_000                      # Questions 3(c), 4(d): Acerbi-Szekely simulated samples
LB_LAGS = 10                        # Ljung-Box H in Questions 1 and 4
NW_LAGS = 10                        # Newey-West truncation lag of the Diebold-Mariano test
SEED = 0                            # fix this, state it in the report, do not change it later

rng = np.random.default_rng(SEED)
print("Project dir:", PROJECT_DIR)


/opt/anaconda3/envs/ada/lib/python3.13/site-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


Project dir: /Users/rachidkahia/Desktop/MA3/QRM/Project-QRM


## 1) Data

The prices were downloaded once with `yfinance` (adjusted closes, 2 January 2013 to 30 June 2026,
downloaded on 28 October 2026) and are provided as `QRM_2026_Project_prices.csv`. Read this file;
do not download the data again, since Yahoo revises adjusted prices back in time.

In [2]:
price_df = pd.read_csv(DATA_FILE, index_col=0, parse_dates=True)[TICKERS]
print(price_df.shape, price_df.index[0].date(), "to", price_df.index[-1].date())   # (3393, 3)

(3393, 3) 2013-01-02 to 2026-06-30


## 2) Log-returns and losses

Construct $R_t=\log(P_t/P_{t-1})$ for each ticker. The log-loss used everywhere in this project is
$L_t=-R_t$. Every question is run separately on each of the three columns; there is no portfolio.

In [3]:
returns = np.log(price_df / price_df.shift(1)).dropna()
losses = -returns

print(returns.shape, returns.index[0].date(), "to", returns.index[-1].date())   # (3392, 3)
returns.head()

(3392, 3) 2013-01-03 to 2026-06-30


,AAPL,META,JPM
Date,,,
2013-01-03,-0.012703,-0.008248,-0.002017
2013-01-04,-0.028250,0.035029,0.017569
2013-01-07,-0.005900,0.022689,0.001101
2013-01-08,0.002688,-0.012312,0.001980
2013-01-09,-0.015752,0.051311,-0.000660


## Question 1 -- Stylized facts

For each stock: (a) plot the return series; (b) ACF of returns and of squared returns, and the
Ljung-Box test with $H$ = `LB_LAGS` on both (`statsmodels.stats.diagnostic.acorr_ljungbox`);
(c) QQ-plot and Jarque-Bera.

In [4]:
# Question 1


## Question 2 -- Single-window VaR and ES estimates

For each stock, use the first `WINDOW` losses as the single estimation window for (a)-(d).

In [5]:
# Question 2
window_loss = losses.iloc[:WINDOW]          # one column per stock


## Question 3 -- Backtesting

Forecast every day of the evaluation period `losses.iloc[EVAL_START:]`, each model estimated on the
`WINDOW` days before the forecast day. Historical simulation, Gaussian and Student-$t$ are recomputed
every day. Write the Kupiec, Christoffersen, Acerbi-Szekely, Fissler-Ziegel, pinball-loss and
Diebold-Mariano functions yourself, from the course slides (Kupiec, Christoffersen) and Appendix A of
the project statement (the others); they are
reused unchanged in Questions 4 and 5.

The combined VaR backtest table collects the VaR backtests of Questions 3-5: append one row per
stock, method and level to `backtest_rows` as you go. The ES backtest table ($\alpha=0.975$) collects
the Acerbi-Szekely and Fissler-Ziegel results of Questions 3 and 4 in `es_rows`.

In [6]:
# Question 3
eval_loss = losses.iloc[EVAL_START:]        # one column per stock, 2888 forecast days
backtest_rows = []                          # dicts: stock, method, alpha, x, expected, hit, p_uc, p_ind, p_cc, pinball_bp
es_rows = []                                # dicts: stock, method, Z, p_Z, FZ


In [7]:
# Q3 : VaR / ES of the 3 methods
alphas = np.array(ALPHAS)
idx = np.ceil(WINDOW * alphas).astype(int) - 1
k = WINDOW - idx - 1


def hs_risk(window):
    Ls = np.sort(window)
    var = Ls[idx]
    ES = np.array([(Ls[i + 1:].sum() + (WINDOW * (1 - a) - j) * Ls[i]) / (WINDOW * (1 - a))
                   for i, j, a in zip(idx, k, alphas)])
    return var, ES


def normal_risk(window):
    mu = np.mean(window)
    sd = np.std(window, ddof=1)
    z = stats.norm.ppf(alphas)
    var = mu + sd * z
    ES = mu + sd * stats.norm.pdf(z) / (1 - alphas)
    return var, ES


def student_risk(df, loc, scale):
    q = stats.t.ppf(alphas, df)
    var = loc + scale * q
    ES = loc + scale * stats.t.pdf(q, df) / (1 - alphas) * (df + q**2) / (df - 1)
    return var, ES


methods = ["HS", "Gaussian", "Student-t"]
VaR_fc = {}
ES_fc = {}
nu_hist = {}

for tk in TICKERS:
    L = losses[tk].values
    for m in methods:
        VaR_fc[(tk, m)] = []
        ES_fc[(tk, m)] = []
    nu_hist[tk] = []

    for t in range(EVAL_START, len(L)):
        window = L[t - WINDOW:t]

        # t fit, also from yesterday's params (local optimum)
        df, loc, scale = stats.t.fit(window)
        if t > EVAL_START:
            df2, loc2, scale2 = stats.t.fit(window, df_old, loc=loc_old, scale=scale_old)
            if stats.t.logpdf(window, df2, loc2, scale2).sum() > stats.t.logpdf(window, df, loc, scale).sum():
                df, loc, scale = df2, loc2, scale2
        df_old, loc_old, scale_old = df, loc, scale
        nu_hist[tk].append(df)

        for m, (var, ES) in zip(methods, [hs_risk(window), normal_risk(window), student_risk(df, loc, scale)]):
            VaR_fc[(tk, m)].append(var)
            ES_fc[(tk, m)].append(ES)

for key in VaR_fc:
    VaR_fc[key] = pd.DataFrame(VaR_fc[key], index=eval_loss.index, columns=ALPHAS)
    ES_fc[key] = pd.DataFrame(ES_fc[key], index=eval_loss.index, columns=ALPHAS)

# first day should match Q2
for tk in TICKERS:
    print(tk, [round(100 * VaR_fc[(tk, m)][0.99].iloc[0], 3) for m in methods])

AAPL [np.float64(3.888), np.float64(3.655), np.float64(3.993)]
META [np.float64(5.348), np.float64(5.64), np.float64(6.034)]
JPM [np.float64(2.778), np.float64(2.623), np.float64(2.889)]


In [8]:
# Q3 : backtest
def backtest_var(VaR, L_test, alpha):
    I = (L_test > VaR).astype(int).values
    n = len(I)
    x = I.sum()
    po = 1 - alpha
    p = x / n

    # Kupiec
    LRuc = 2 * (x * np.log(p / po) + (n - x) * np.log((1 - p) / (1 - po)))

    # Christoffersen
    prev, curr = I[:-1], I[1:]
    n00 = np.sum((prev == 0) & (curr == 0))
    n01 = np.sum((prev == 0) & (curr == 1))
    n10 = np.sum((prev == 1) & (curr == 0))
    n11 = np.sum((prev == 1) & (curr == 1))

    p01 = n01 / (n00 + n01)
    p11 = n11 / (n10 + n11)
    p_all = (n01 + n11) / (n00 + n01 + n10 + n11)

    l0 = (n01 + n11) * np.log(p_all) + (n00 + n10) * np.log(1 - p_all)
    l1 = n01 * np.log(p01) + n00 * np.log(1 - p01)
    if n11 > 0:
        l1 += n11 * np.log(p11)
    if n10 > 0:
        l1 += n10 * np.log(1 - p11)

    LRind = -2 * (l0 - l1)
    LRcc = LRuc + LRind

    # pinball
    pin = ((L_test <= VaR) - alpha) * (VaR - L_test)

    return {"x": x,
            "expected": n * po,
            "hit": p,
            "p_uc": 1 - stats.chi2.cdf(LRuc, 1),
            "p_ind": 1 - stats.chi2.cdf(LRind, 1),
            "p_cc": 1 - stats.chi2.cdf(LRcc, 2),
            "pinball_bp": 1e4 * pin.mean()}

In [9]:
# Q3 (a)
for (tk, m), VaR in VaR_fc.items():
    for alpha in ALPHAS:
        res = backtest_var(VaR[alpha], eval_loss[tk], alpha)
        backtest_rows.append({"stock": tk, "method": m, "alpha": alpha, **res})

table_q3 = pd.DataFrame(backtest_rows).set_index(["alpha", "stock", "method"]).sort_index()
table_q3.to_csv(OUT_DIR / "q3_var_backtest.csv")

for alpha in ALPHAS:
    print("alpha =", alpha)
    display(table_q3.loc[alpha].round(3))

alpha = 0.975


x  expected    hit   p_uc  p_ind   p_cc  pinball_bp
stock method                                                          
AAPL  Gaussian   105      72.2  0.036  0.000  0.000  0.000      13.748
      HS          86      72.2  0.030  0.110  0.004  0.005      13.601
      Student-t  103      72.2  0.036  0.001  0.000  0.000      13.716
JPM   Gaussian    95      72.2  0.033  0.009  0.000  0.000      13.207
      HS          83      72.2  0.029  0.208  0.000  0.000      13.023
      Student-t   92      72.2  0.032  0.023  0.000  0.000      13.303
META  Gaussian    78      72.2  0.027  0.495  0.000  0.001      18.321
      HS          75      72.2  0.026  0.740  0.016  0.051      18.262
      Student-t   89      72.2  0.031  0.053  0.000  0.000      18.285

alpha = 0.99


x  expected    hit   p_uc  p_ind   p_cc  pinball_bp
stock method                                                         
AAPL  Gaussian   62     28.88  0.021  0.000  0.012  0.000       7.658
      HS         41     28.88  0.014  0.033  0.138  0.034       7.244
      Student-t  41     28.88  0.014  0.033  0.138  0.034       7.211
JPM   Gaussian   64     28.88  0.022  0.000  0.000  0.000       7.581
      HS         42     28.88  0.015  0.022  0.000  0.000       7.216
      Student-t  47     28.88  0.016  0.002  0.000  0.000       7.353
META  Gaussian   52     28.88  0.018  0.000  0.002  0.000      10.672
      HS         40     28.88  0.014  0.049  0.125  0.045      10.372
      Student-t  38     28.88  0.013  0.104  0.102  0.070      10.501

In [10]:
# Q3 (b)
verdict = table_q3[["x", "expected", "p_uc", "p_ind", "p_cc"]].copy()
verdict["fail_uc"] = verdict["p_uc"] < 0.05
verdict["fail_ind"] = verdict["p_ind"] < 0.05
verdict["fail_cc"] = verdict["p_cc"] < 0.05
display(verdict.round(3))

# Kupiec and cc disagree
display(verdict[verdict["fail_uc"] != verdict["fail_cc"]].round(3))

x  expected   p_uc  p_ind   p_cc  fail_uc  fail_ind  \
alpha stock method                                                             
0.975 AAPL  Gaussian   105     72.20  0.000  0.000  0.000     True      True   
            HS          86     72.20  0.110  0.004  0.005    False      True   
            Student-t  103     72.20  0.001  0.000  0.000     True      True   
      JPM   Gaussian    95     72.20  0.009  0.000  0.000     True      True   
            HS          83     72.20  0.208  0.000  0.000    False      True   
            Student-t   92     72.20  0.023  0.000  0.000     True      True   
      META  Gaussian    78     72.20  0.495  0.000  0.001    False      True   
            HS          75     72.20  0.740  0.016  0.051    False      True   
            Student-t   89     72.20  0.053  0.000  0.000    False      True   
0.990 AAPL  Gaussian    62     28.88  0.000  0.012  0.000     True      True   
            HS          41     28.88  0.033  0.138  0.034     True     False   
            Student-t   41     28.88  0.033  0.138  0.034     True     False   
      JPM   Gaussian    64     28.88  0.000  0.000  0.000     True      True   
            HS          42     28.88  0.022  0.000  0.000     True      True   
            Student-t   47     28.88  0.002  0.000  0.000     True      True   
      META  Gaussian    52     28.88  0.000  0.002  0.000     True      True   
            HS          40     28.88  0.049  0.125  0.045     True     False   
            Student-t   38     28.88  0.104  0.102  0.070    False     False   

                       fail_cc  
alpha stock method              
0.975 AAPL  Gaussian      True  
            HS            True  
            Student-t     True  
      JPM   Gaussian      True  
            HS            True  
            Student-t     True  
      META  Gaussian      True  
            HS           False  
            Student-t     True  
0.990 AAPL  Gaussian      True  
            HS            True  
            Student-t     True  
      JPM   Gaussian      True  
            HS            True  
            Student-t     True  
      META  Gaussian      True  
            HS            True  
            Student-t    False

x  expected   p_uc  p_ind   p_cc  fail_uc  fail_ind  \
alpha stock method                                                            
0.975 AAPL  HS         86      72.2  0.110  0.004  0.005    False      True   
      JPM   HS         83      72.2  0.208  0.000  0.000    False      True   
      META  Gaussian   78      72.2  0.495  0.000  0.001    False      True   
            Student-t  89      72.2  0.053  0.000  0.000    False      True   

                       fail_cc  
alpha stock method              
0.975 AAPL  HS            True  
      JPM   HS            True  
      META  Gaussian      True  
            Student-t     True

## Question 4 -- AR-GARCH conditional VaR and ES

(a) Mean model, full sample: fit AR($p$), $p=0,\dots,3$, with constant variance on the same
observations, e.g. `ARX(y, lags=p, hold_back=3)` from `arch.univariate`; pick $p$ by BIC (also report
AIC); Ljung-Box on the residuals (`model_df=p`) and on their squares.
(b) Fit `arch_model(y, mean="AR", lags=p, vol="GARCH", p=1, q=1, dist=..., hold_back=3)` on the full
sample with `dist="normal"` and `dist="t"` (use `mean="Constant"` if $p=0$); Ljung-Box on the
standardized residuals of the Gaussian fit (`model_df=p`) and their squares (`model_df=2`).
(c) Re-estimate both inside the rolling scheme every `REFIT` days with the same $p$. Between refits,
keep the parameters fixed but advance $\hat\mu_t$ and $\hat\sigma_t$ daily with the realized
returns (for example with `arch_model(...).fix(params)` on the data up to the forecast day). Fit on
returns in percent (`100 * R`) to avoid `arch` convergence warnings, and convert the forecasts back.
(d) Backtest both and add them to the two tables.

In [ ]:
# Question 4
from arch import arch_model


## Question 5 -- Quantile regression

Same evaluation period, window and refit days as Question 4. One model per stock and level, VaR only;
the two features are built by `make_features` below, both using information up to $t-1$; the target
is the loss $L_t$. Rows with an undefined feature (the first 20 days) are dropped from the training
window. Linear quantile regression, e.g.
`sm.QuantReg(y, sm.add_constant(X)).fit(q=alpha, max_iter=5000)`.

In [ ]:
# Question 5
import statsmodels.api as sm


def make_features(ret: pd.Series) -> pd.DataFrame:
    """Features x_{t-1} for the forecast of day t: row t uses information up to t-1 only."""
    feat = pd.DataFrame(index=ret.index)
    feat["absR_lag1"] = ret.shift(1).abs()
    feat["sigma20"] = ret.shift(1).rolling(20).std(ddof=1)
    return feat


# After (a) and (b): the combined VaR backtest table, one table per level
# backtest_table = pd.DataFrame(backtest_rows).set_index(["alpha", "stock", "method"]).sort_index()


## Question 6 -- Extreme value theory

One static fit on each stock's full-sample log-loss series, with threshold $u$ the empirical 95%
quantile. `scipy.stats.genpareto.fit(excesses, floc=0)` may be used for the GPD maximum-likelihood
step.

In [ ]:
# Question 6
from scipy.stats import genpareto
full_loss = losses                          # one column per stock
